In [1]:
# Standard library
import datetime
import json
import random
import time
from argparse import ArgumentParser
from dataclasses import dataclass, field
from typing import List

# Third-party
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.profilers import AdvancedProfiler

# First-party
from neural_lam import constants, utils, config
from neural_lam.weather_dataset import WeatherDataset
from neural_lam.downscaling_dataset import DownscalingDataset
from neural_lam.netCDF_dataset import NetCDFDataset
from neural_lam.models.graph_efm import GraphEFM
from neural_lam.models.graph_fm import GraphFM
from neural_lam.models.graphcast import GraphCast
from neural_lam.models.diffusion import Diffusion
from neural_lam.models.ir_sde import IR_SDE
from neural_lam.models.stochastic_interpolants import SI

In [2]:
MODELS = {
    "graphcast": GraphCast,
    "graph_fm": GraphFM,
    "graph_efm": GraphEFM,
    "diffusion": Diffusion,
    "ir_sde": IR_SDE, 
    "SI": SI,
}

config_loader = config.Config.from_file("/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/neural_lam/clim_config_inference.yaml")
random_run_id = random.randint(0, 9999)
seed.seed_everything(42)
# Instantiate model + trainer
if torch.cuda.is_available():
    device_name = "cuda"
    torch.set_float32_matmul_precision(
        "high"
    )  # Allows using Tensor Cores on A100s
else:
    device_name = "cpu"

print(f"Using device: {device_name}")
model_class = MODELS["SI"]

[rank: 0] Seed set to 42


Using device: cuda


In [3]:
@dataclass
class SIModelArguments:
    load: str = "/mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/saved_models/SI_50e-SI-6x128-07_10_12-7283/last.ckpt"
    data_config: str = "/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/neural_lam/clim_config_inference.yaml"
    ensemble_size: int = 4  # --ensemble_size
    n_example_pred: int = 99999 # set it to a larger number so that we always output examples
    sampler_steps: int = 2  # --sampler_steps
    sampler: str = "euler"  # --sampler
    save_output: bool = True  # not set → False
    output_path: str = "output"
    save_output_wandb: bool = False  # not set → False
    save_steps: bool = False  # not set → False
    output_std: bool = False
    subset_ds: bool = False
    n_workers: int = 16
    batch_size: int = 4
    loss: str = "wmse"
    step_length: int = 3
    lr: float = 1E-3
    lr_scheduler: str = "None"
    weight_decay: float = 1E-2
    restore_opt: bool = False
    pred_residual: bool = False  # not set → False
    diffusion_model: str = "song_unet"  # --diffusion_model
    noise_embedding: str = "fourier"  # default
    resample_filter: List[int] = field(default_factory=lambda: [1, 1])  # default
    channel_mult: List[int] = field(default_factory=lambda: [1, 2, 2, 2])  # default
    encoder_type: str = "standard"  # default
    attn_resolutions: List[int] = field(default_factory=lambda: [1])  # default
    sigma_coef: float = 1.0  # default
    sigma_max: float = 10 / 255
    sigma_min: float = 0.002
    wandb_project: str = "clim-downscaling"

In [4]:
args = SIModelArguments()
model = model_class(args)

res: tensor([400, 550])
Attn at level 0 with res 400: False
Attn at level 0 with res 400: False
Attn at level 0 with res 400: False
Attn at level 0 with res 400: False
Channels at level 0: 128
res_down: tensor([200, 276], dtype=torch.int32)
Attn at level 1 with res 200: False
Attn at level 1 with res 200: False
Attn at level 1 with res 200: False
Attn at level 1 with res 200: False
Channels at level 1: 256
res_down: tensor([100, 138], dtype=torch.int32)
Attn at level 2 with res 100: False
Attn at level 2 with res 100: False
Attn at level 2 with res 100: False
Attn at level 2 with res 100: False
Channels at level 2: 256
res_down: tensor([50, 70], dtype=torch.int32)
Attn at level 3 with res 50: False
Attn at level 3 with res 50: False
Attn at level 3 with res 50: False
Attn at level 3 with res 50: False
Channels at level 3: 256
res_up: tensor([100, 138], dtype=torch.int32)
res_up: tensor([200, 276], dtype=torch.int32)
res_up: tensor([400, 550], dtype=torch.int32)


In [5]:
eval_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.validation_start_date,
        end_date=config_loader.dataset.validation_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=args.subset_ds,
    ),
    args.batch_size,
    shuffle=False,
    num_workers=args.n_workers,
    pin_memory=True,
    persistent_workers=True,
)

input_files: ['standardized.clt_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.hus_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.pr_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_mm_day_noleap.nc', 'standardized.psl_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.tas_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ta_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ua_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.va_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc']


In [8]:
logger = pl.loggers.WandbLogger(
    project=args.wandb_project, name='test_inference', config=args
)

trainer = pl.Trainer(
    max_epochs=1,
    deterministic=True,
    accelerator=device_name,
    logger=logger,
    log_every_n_steps=1,
    check_val_every_n_epoch=1,
    precision=32,
    profiler="simple",
)

trainer.test(model=model, dataloaders=eval_loader, ckpt_path=args.load)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id q9ouko51.


Restoring states from the checkpoint path at /mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/saved_models/SI_50e-SI-6x128-07_10_12-7283/last.ckpt
/opt/conda/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:282: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["ModelCheckpoint{'monitor': 'val_mean_loss', 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}"].
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /mimer/NOBACKUP/groups/mlhighres/users/erifh/neural-lam/saved_models/SI_50e-SI-6x128-07_10_12-7283/last.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

Saving 2009-01-01 sample to disk...
Saving 2009-01-02 sample to disk...
Saving 2009-01-03 sample to disk...
Saving 2009-01-04 sample to disk...
Saving 2009-01-05 sample to disk...
Saving 2009-01-06 sample to disk...
Saving 2009-01-07 sample to disk...
Saving 2009-01-08 sample to disk...
Saving 2009-01-09 sample to disk...
Saving 2009-01-10 sample to disk...
Saving 2009-01-11 sample to disk...
Saving 2009-01-12 sample to disk...



Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined